# S9 - Observabilidad del pipeline `red-telemetria` (Proyecto Sello)

Actividad autónoma de S9 (sección 4.1) sobre el topic `red-telemetria` de S7-S8, que alimenta en vivo el agente
`uso-red/agente/agente_red.py` con los contadores reales de las tarjetas de red de esta PC.

Antes de ejecutar: Kafka, `uso-red` (bridge y `consumer_red.py`, que expone `/metrics` en el puerto 9200), el agente,
`obs/` (Prometheus y Grafana) y `pyspark` con `compose.kafka.yml`.

| Tarea 4.1 | Sección |
|---|---|
| 1. Lectura streaming con `isValid` / `processedAt` / `latencyMs` | 1-2 |
| 2-3. Prometheus (targets `UP`) y Grafana | 5 (consultas desde el notebook) + `obs/` |
| 4. Latencia agregada por ventana sobre la evidencia persistida | 4 |
| 5. Alertas con umbrales justificados por mis datos | 6 |
| 6. Estimación de costos y plan de escalado | 7 |

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("s09-observabilidad-red-telemetria")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "3")
    .config("spark.sql.session.timeZone", "America/Lima")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6a72e916-14e4-4147-ae88-8af0b247ef82;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central


	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central
	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 259ms :: artifacts dl 9ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central i

26/10/16 01:27:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.2.0


In [2]:
import json, os, shutil, time, urllib.parse, urllib.request
from datetime import datetime, timedelta, timezone
from pyspark.sql.functions import (avg, col, count, current_timestamp, expr, from_json, greatest,
                                   max as smax, min as smin, percentile_approx, round as sround,
                                   unix_millis, when, window)
from pyspark.sql.types import DoubleType, LongType, StringType, StructType

BOOTSTRAP, TOPIC, BASE = "kafka:9092", "red-telemetria", "./artifacts"
PROMETHEUS = "http://prometheus:9090"
LIMA = timezone(timedelta(hours=-5))

def hora(ms=None):
    return datetime.fromtimestamp((ms or time.time() * 1000) / 1000, LIMA).strftime("%H:%M:%S")

def tam_mb(ruta):
    return sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(ruta) for f in fs if f.endswith(".parquet")) / 1e6

## 1. Lectura streaming con campos de observabilidad

In [3]:
esquema_red = (StructType()
    .add("tipoEvento", StringType()).add("sensorId", StringType()).add("host", StringType())
    .add("interfaz", StringType()).add("tipoInterfaz", StringType())
    .add("velocidadEnlaceMbps", DoubleType()).add("intervaloMs", LongType())
    .add("bytesRxPorSeg", DoubleType()).add("bytesTxPorSeg", DoubleType())
    .add("paquetesRxPorSeg", DoubleType()).add("paquetesTxPorSeg", DoubleType())
    .add("erroresPorSeg", DoubleType()).add("descartesPorSeg", DoubleType())
    .add("origen", StringType()).add("timestamp", LongType()))

crudo = (spark.readStream.format("kafka")
         .option("kafka.bootstrap.servers", BOOTSTRAP)
         .option("subscribe", TOPIC)
         .option("startingOffsets", "latest")
         .load())

eventos = (crudo
    .select("topic", "partition", "offset", col("timestamp").alias("kafkaTimestamp"),
            from_json(col("value").cast("string"), esquema_red).alias("e"))
    .select("topic", "partition", "offset", "kafkaTimestamp", "e.*"))

observables = (eventos
    # isValid: el mismo contrato de esquema que consumer_red.py (S7), calculado en Spark
    .withColumn("isValid", (col("tipoEvento") == "red.lectura") & col("sensorId").isNotNull()
                & col("velocidadEnlaceMbps").isNotNull() & col("bytesRxPorSeg").isNotNull()
                & col("bytesTxPorSeg").isNotNull() & col("timestamp").isNotNull())
    .withColumn("fueraDeRango", col("isValid") & (
        (col("bytesRxPorSeg") < 0) | (col("bytesTxPorSeg") < 0) |
        ((col("velocidadEnlaceMbps") > 0) &
         (greatest("bytesRxPorSeg", "bytesTxPorSeg") > col("velocidadEnlaceMbps") * 125000 * 1.05))))
    .withColumn("processedAt", unix_millis(current_timestamp()))
    .withColumn("latencyMs", col("processedAt") - col("timestamp"))
    # transporteMs: solo el viaje agente -> MQTT -> bridge -> broker, sin la espera del trigger
    .withColumn("transporteMs", unix_millis(col("kafkaTimestamp")) - col("timestamp")))

observables.printSchema()

root
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- kafkaTimestamp: timestamp (nullable = true)
 |-- tipoEvento: string (nullable = true)
 |-- sensorId: string (nullable = true)
 |-- host: string (nullable = true)
 |-- interfaz: string (nullable = true)
 |-- tipoInterfaz: string (nullable = true)
 |-- velocidadEnlaceMbps: double (nullable = true)
 |-- intervaloMs: long (nullable = true)
 |-- bytesRxPorSeg: double (nullable = true)
 |-- bytesTxPorSeg: double (nullable = true)
 |-- paquetesRxPorSeg: double (nullable = true)
 |-- paquetesTxPorSeg: double (nullable = true)
 |-- erroresPorSeg: double (nullable = true)
 |-- descartesPorSeg: double (nullable = true)
 |-- origen: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- isValid: boolean (nullable = true)
 |-- fueraDeRango: boolean (nullable = true)
 |-- processedAt: long (nullable = false)
 |-- latencyMs: long (nullable = true)
 |-- transporteM

## 2. Salida doble (consola y Parquet) durante 7 minutos, con `lastProgress` cada 30 s

La consulta "consola" usa `foreachBatch` para mostrar sus micro-lotes dentro del notebook (el sink `console` imprime en
el log del contenedor). La consulta Parquet persiste la evidencia con su propio checkpoint. Cada 30 s se registra
`lastProgress` en formato de log estructurado y el tamaño del Parquet.

In [4]:
PARQUET = f"{BASE}/red_observabilidad_parquet"
CHK_PARQUET = f"{BASE}/chk_observabilidad"
for ruta in (PARQUET, CHK_PARQUET, f"{BASE}/chk_consola"):
    shutil.rmtree(ruta, ignore_errors=True)

COLUMNAS = ["partition", "offset", "sensorId", "bytesRxPorSeg", "isValid", "fueraDeRango",
            "timestamp", "processedAt", "latencyMs", "transporteMs"]
lotes_consola, resumen_lotes = [], []

def a_consola(batch_df, batch_id):
    # Una acción en CADA micro-lote: si foreachBatch no toca batch_df, Spark no lee de Kafka
    # (evaluación perezosa) y lastProgress reporta numInputRows = 0 aunque lleguen datos.
    pdf = batch_df.select(*COLUMNAS).toPandas()
    resumen_lotes.append((batch_id, len(pdf), pdf["latencyMs"].mean() if len(pdf) else None))
    if len(lotes_consola) < 4:
        lotes_consola.append((batch_id, pdf))

consulta = (observables.writeStream.outputMode("append").foreachBatch(a_consola)
            .option("checkpointLocation", f"{BASE}/chk_consola")
            .trigger(processingTime="5 seconds").start())
consulta2 = (observables.writeStream.outputMode("append").format("parquet")
             .option("path", PARQUET).option("checkpointLocation", CHK_PARQUET)
             .trigger(processingTime="5 seconds").start())

def log_progreso(progreso):
    metricas = progreso["sources"][0].get("metrics", {})
    return {
        "service": "spark-streaming", "component": "observabilidad", "topic": TOPIC,
        "hora": hora(), "batchId": progreso["batchId"], "numInputRows": progreso["numInputRows"],
        "inputRowsPerSecond": round(progreso["inputRowsPerSecond"], 2),
        "processedRowsPerSecond": round(progreso["processedRowsPerSecond"], 2),
        "batchDurationMs": progreso["durationMs"].get("triggerExecution"),
        "avgOffsetsBehindLatest": metricas.get("avgOffsetsBehindLatest"),
        "maxOffsetsBehindLatest": metricas.get("maxOffsetsBehindLatest"),
        "status": "idle" if progreso["numInputRows"] == 0 else "processed",
    }

muestras, crecimiento = [], []
inicio = time.time()
while time.time() - inicio < 420:
    time.sleep(30)
    if consulta.lastProgress:
        muestras.append(log_progreso(consulta.lastProgress))
    try:
        filas = spark.read.parquet(PARQUET).count()
    except Exception:
        filas = 0
    crecimiento.append({"segundos": round(time.time() - inicio), "hora": hora(), "filas": filas, "mb": round(tam_mb(PARQUET), 4)})

consulta.stop()
consulta2.stop()
print(f"Consultas detenidas a las {hora()}: {len(muestras)} muestras de lastProgress, {len(crecimiento)} mediciones del Parquet")

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


...

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Consultas detenidas a las 20:34:38: 14 muestras de lastProgress, 14 mediciones del Parquet


In [5]:
import pandas as pd
pd.set_option("display.width", 250)
for batch_id, pdf in lotes_consola[1:3]:
    print(f"-------------------- Batch {batch_id} --------------------")
    print(pdf.head(8).to_string(index=False))

-------------------- Batch 1 --------------------
 partition  offset        sensorId  bytesRxPorSeg  isValid  fueraDeRango     timestamp   processedAt  latencyMs  transporteMs
         1    2092 nsmj-ethernet-2            0.0     True         False 1792114037604 1792114040008       2404             3
         1    2093 nsmj-ethernet-2            0.0     True         False 1792114038644 1792114040008       1364             2
         1    2094 nsmj-ethernet-2            0.0     True         False 1792114039682 1792114040008        326             3
         0    4194   nsmj-ethernet      4254570.3     True         False 1792114037604 1792114040008       2404             1
         0    4195  nsmj-tailscale          601.4     True         False 1792114037605 1792114040008       2403             4
         0    4196   nsmj-ethernet       124686.7     True         False 1792114038644 1792114040008       1364             1
         0    4197  nsmj-tailscale          296.1     True         F

In [6]:
print("lastProgress de la consulta (último micro-lote), como log estructurado:")
print(json.dumps(muestras[-1], indent=2))
print()
print(pd.DataFrame(muestras)[["hora", "batchId", "numInputRows", "inputRowsPerSecond", "processedRowsPerSecond",
                              "batchDurationMs", "avgOffsetsBehindLatest", "maxOffsetsBehindLatest", "status"]].to_string(index=False))

lastProgress de la consulta (último micro-lote), como log estructurado:
{
  "service": "spark-streaming",
  "component": "observabilidad",
  "topic": "red-telemetria",
  "hora": "20:34:36",
  "batchId": 88,
  "numInputRows": 20,
  "inputRowsPerSecond": 4.0,
  "processedRowsPerSecond": 36.0,
  "batchDurationMs": 556,
  "avgOffsetsBehindLatest": "0.0",
  "maxOffsetsBehindLatest": "0",
  "status": "processed"
}

    hora  batchId  numInputRows  inputRowsPerSecond  processedRowsPerSecond  batchDurationMs avgOffsetsBehindLatest maxOffsetsBehindLatest    status
20:27:46        6            20                 4.0                    37.3              536                    0.0                      0 processed
20:28:17       12            20                 4.0                    59.3              337                    0.0                      0 processed
20:28:48       18            20                 4.0                    26.0              768                    0.0                      0 p

## 3. Crecimiento del Parquet

In [7]:
crec = pd.DataFrame(crecimiento)
print(crec.to_string(index=False))
a, b = crecimiento[1], crecimiento[-1]
filas_h = (b["filas"] - a["filas"]) / (b["segundos"] - a["segundos"]) * 3600
mb_h = (b["mb"] - a["mb"]) / (b["segundos"] - a["segundos"]) * 3600
print(f"\nCrecimiento medido entre {a['hora']} y {b['hora']}: {filas_h:,.0f} filas/hora, {mb_h:.2f} MB/hora")

 segundos     hora  filas     mb
       31 20:27:47    108 0.1418
       61 20:28:18    224 0.2813
       92 20:28:49    344 0.4209
      123 20:29:19    460 0.5603
      154 20:29:50    576 0.7230
      185 20:30:22    692 0.8625
      217 20:30:54    828 1.0019
      249 20:31:25    944 1.1415
      280 20:31:57   1060 1.3042
      312 20:32:29   1200 1.4438
      344 20:33:01   1316 1.6066
      377 20:33:33   1432 1.7460
      409 20:34:06   1568 1.8854
      442 20:34:38   1684 2.0479

Crecimiento medido entre 20:28:18 y 20:34:38: 13,795 filas/hora, 16.69 MB/hora


## 4. Latencia agregada por ventana de 1 minuto (lectura batch sobre la evidencia persistida)

In [8]:
guardado = spark.read.parquet(PARQUET)
print("filas guardadas:", guardado.count(), "| válidas:", guardado.filter("isValid").count(),
      "| fuera de rango:", guardado.filter("fueraDeRango").count())

latencia_por_minuto = (guardado.groupBy(window(col("kafkaTimestamp"), "1 minute"))
    .agg(count("*").alias("numEventos"),
         sround(avg("latencyMs"), 1).alias("avgLatencyMs"),
         smin("latencyMs").alias("minLatencyMs"),
         smax("latencyMs").alias("maxLatencyMs"),
         percentile_approx("latencyMs", 0.95).alias("p95LatencyMs"),
         sround(avg("transporteMs"), 1).alias("avgTransporteMs"))
    .select(expr("date_format(window.start, 'HH:mm')").alias("inicio"),
            expr("date_format(window.end, 'HH:mm')").alias("fin"),
            "numEventos", "avgLatencyMs", "minLatencyMs", "maxLatencyMs", "p95LatencyMs", "avgTransporteMs")
    .orderBy("inicio"))
latencia_por_minuto.show(20, truncate=False)

filas guardadas: 1704 | válidas: 1704 | fuera de rango: 0


+------+-----+----------+------------+------------+------------+------------+---------------+
|inicio|fin  |numEventos|avgLatencyMs|minLatencyMs|maxLatencyMs|p95LatencyMs|avgTransporteMs|
+------+-----+----------+------------+------------+------------+------------+---------------+
|20:27 |20:28|168       |2347.8      |35          |4871        |4592        |2.4            |
|20:28 |20:29|232       |2485.5      |14          |4917        |4789        |2.7            |
|20:29 |20:30|232       |2445.0      |51          |4947        |4617        |2.5            |
|20:30 |20:31|236       |2607.4      |193         |5003        |4867        |2.4            |
|20:31 |20:32|232       |2538.2      |28          |4994        |4877        |2.2            |
|20:32 |20:33|236       |2441.3      |29          |4897        |4665        |2.6            |
|20:33 |20:34|232       |2504.4      |16          |4958        |4780        |2.5            |
|20:34 |20:35|136       |2515.2      |88          |4955     

In [9]:
resumen = guardado.agg(
    percentile_approx("latencyMs", 0.5).alias("p50"), percentile_approx("latencyMs", 0.95).alias("p95"),
    smax("latencyMs").alias("max"), percentile_approx("transporteMs", 0.5).alias("transp_p50"),
    percentile_approx("transporteMs", 0.99).alias("transp_p99"), smax("transporteMs").alias("transp_max")).first()
print(f"latencyMs   (sensor -> Spark): p50={resumen.p50} ms  p95={resumen.p95} ms  max={resumen.max} ms")
print(f"transporteMs (sensor -> broker): p50={resumen.transp_p50} ms  p99={resumen.transp_p99} ms  max={resumen.transp_max} ms")

latencyMs   (sensor -> Spark): p50=2489 ms  p95=4753 ms  max=5003 ms
transporteMs (sensor -> broker): p50=2 ms  p99=5 ms  max=8 ms


### 4.1 Capacidad: cuánto procesa Spark con backlog

Para el plan de escalado hace falta saber cuántas filas por segundo procesa este Spark local cuando tiene trabajo
acumulado, no solo a 4 eventos/s. Se lee todo el topic desde `earliest` con `trigger(availableNow=True)`.

In [10]:
shutil.rmtree(f"{BASE}/chk_capacidad", ignore_errors=True)
crudo_todo = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", BOOTSTRAP)
              .option("subscribe", TOPIC).option("startingOffsets", "earliest")
              .option("maxOffsetsPerTrigger", 2000).load())
obs_todo = (crudo_todo.select(col("timestamp").alias("kafkaTimestamp"),
                              from_json(col("value").cast("string"), esquema_red).alias("e"))
            .select("kafkaTimestamp", "e.*")
            .withColumn("processedAt", unix_millis(current_timestamp()))
            .withColumn("latencyMs", col("processedAt") - col("timestamp")))
q = (obs_todo.writeStream.format("noop").option("checkpointLocation", f"{BASE}/chk_capacidad")
     .trigger(availableNow=True).start())
q.awaitTermination()
lotes = [p for p in q.recentProgress if p["numInputRows"] > 0]
for p in lotes:
    print(f"batchId={p['batchId']}  filas={p['numInputRows']}  processedRowsPerSecond={p['processedRowsPerSecond']:.0f}")
capacidad = max(p["processedRowsPerSecond"] for p in lotes)
print(f"\nCapacidad medida (mejor micro-lote): {capacidad:,.0f} filas/s")

batchId=0  filas=2000  processedRowsPerSecond=4435
batchId=1  filas=2000  processedRowsPerSecond=7812
batchId=2  filas=1116  processedRowsPerSecond=1449

Capacidad medida (mejor micro-lote): 7,812 filas/s


## 5. Prometheus desde el notebook: targets y consultas PromQL reales

In [11]:
def promql(consulta):
    url = f"{PROMETHEUS}/api/v1/query?" + urllib.parse.urlencode({"query": consulta})
    return json.load(urllib.request.urlopen(url, timeout=10))["data"]["result"]

objetivos = json.load(urllib.request.urlopen(f"{PROMETHEUS}/api/v1/targets", timeout=10))["data"]["activeTargets"]
for t in objetivos:
    print(f"target {t['labels']['job']:<15} {t['health'].upper():<5} {t['scrapeUrl']}")

consultas = {
    "kafka_brokers": "kafka_brokers",
    "throughput actual (msg/s)": 'sum(rate(kafka_topic_partition_current_offset{topic="red-telemetria"}[1m]))',
    "lag actual uso-red-group": 'sum(kafka_consumergroup_lag{consumergroup="uso-red-group"})',
    "lag máximo en 30 min": 'max_over_time(sum(kafka_consumergroup_lag{consumergroup="uso-red-group"})[30m:15s])',
    "lag mediano en 30 min": 'quantile_over_time(0.5, sum(kafka_consumergroup_lag{consumergroup="uso-red-group"})[30m:15s])',
    "lecturas consumed (total)": 'sum(uso_red_eventos_total{status="consumed"})',
    "latencia consumer p95 (ms, 5m)": "histogram_quantile(0.95, sum by (le) (rate(uso_red_latencia_ms_bucket[5m])))",
    "uso máximo del enlace en 30 min (%)": 'max_over_time(max(uso_red_uso_enlace_pct)[30m:15s])',
}
medido = {}
for nombre, q_ in consultas.items():
    r = promql(q_)
    medido[nombre] = float(r[0]["value"][1]) if r else None
    print(f"{nombre:<42} {medido[nombre]}")

# Lag durante la prueba real: consumer_red.py detenido de 20:13:54 a 20:16:55 (hora de Lima).
# Se acota el rango para no mezclarlo con el lag fantasma de las 20:12 (ver Error o hallazgo del informe).
def rango_lima(texto):
    return datetime.strptime(f"2026-10-15 {texto}", "%Y-%m-%d %H:%M:%S").replace(tzinfo=LIMA).timestamp()

url = f"{PROMETHEUS}/api/v1/query_range?" + urllib.parse.urlencode({
    "query": 'sum(kafka_consumergroup_lag{consumergroup="uso-red-group"})',
    "start": rango_lima("20:13:30"), "end": rango_lima("20:18:00"), "step": 15})
serie = json.load(urllib.request.urlopen(url, timeout=10))["data"]["result"][0]["values"]
medido["lag máximo en la prueba (consumer detenido 3 min)"] = max(float(v) for _, v in serie)
print(f"{'lag máximo en la prueba (consumer detenido 3 min)':<42} {medido['lag máximo en la prueba (consumer detenido 3 min)']}")
print("serie 20:13:30-20:18:00:", [(hora(t * 1000), int(float(v))) for t, v in serie][::2])

target kafka-exporter  UP    http://kafka-exporter:9308/metrics
target prometheus      UP    http://localhost:9090/metrics
target uso-red         UP    http://uso-red:9200/metrics
kafka_brokers                              1.0
throughput actual (msg/s)                  3.911198026622814
lag actual uso-red-group                   4.0
lag máximo en 30 min                       4549.0
lag mediano en 30 min                      8.0
lecturas consumed (total)                  4880.0
latencia consumer p95 (ms, 5m)             4.909514925373134
uso máximo del enlace en 30 min (%)        46.98
lag máximo en la prueba (consumer detenido 3 min) 688.0
serie 20:13:30-20:18:00: [('20:13:30', 4), ('20:14:00', 8), ('20:14:30', 104), ('20:15:00', 220), ('20:15:30', 336), ('20:16:00', 452), ('20:16:30', 572), ('20:17:00', 688), ('20:17:30', 8), ('20:18:00', 4)]


## 6. Alertas propuestas, con umbrales calculados a partir de lo medido

In [12]:
lat_min = latencia_por_minuto.toPandas()
peor_avg = lat_min["avgLatencyMs"].max()
peor_max = lat_min["maxLatencyMs"].max()
eventos_s = 4   # 4 sensores x 1 lectura/s (intervalo del agente)

alertas = [
    ("Kafka no visible", "kafka_brokers < 1", "1m",
     f"medido: {medido['kafka_brokers']:.0f} broker; sin broker no hay ingesta"),
    ("Target caído (exporter o consumer)", 'up{job=~"kafka-exporter|uso-red"} == 0', "1m",
     "medido: 1 en ambos; un scrape fallido es binario"),
    ("Sensor o agente detenido", 'sum(rate(kafka_topic_partition_current_offset{topic="red-telemetria"}[1m])) < 3', "2m",
     f"esperado {eventos_s} msg/s; < 3 significa que dejó de publicar al menos 1 de 4 sensores"),
    ("Lag alto sostenido", 'sum(kafka_consumergroup_lag{consumergroup="uso-red-group"}) > 120', "2m",
     f"mediana normal = {medido['lag mediano en 30 min']:.0f}; 120 = 30 s de producción ({eventos_s} msg/s x 30 s); con el consumer detenido 3 min llegó a {medido['lag máximo en la prueba (consumer detenido 3 min)']:.0f} (unas 6 veces el umbral)"),
    ("Latencia alta sensible", "avgLatencyMs por minuto > 6000", "1 ventana",
     f"peor avg/min medido = {peor_avg:.0f} ms (máx {peor_max} ms); trigger 5 s + transporte < 50 ms => > 6 s ya no es la espera del trigger"),
    ("Latencia alta crítica", "avgLatencyMs por minuto > 15000", "1 ventana",
     "3 triggers de retraso: Spark no alcanza a procesar a tiempo"),
    ("Saturación del enlace (dominio)", "max(uso_red_uso_enlace_pct) > 70", "1m",
     f"umbral de S7; uso máximo medido hoy = {medido['uso máximo del enlace en 30 min (%)']:.2f} %"),
]
print(pd.DataFrame(alertas, columns=["Situación", "Regla", "Durante", "Justificación con datos"]).to_string(index=False))

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


                         Situación                                                                           Regla   Durante                                                                                                            Justificación con datos
                  Kafka no visible                                                               kafka_brokers < 1        1m                                                                                        medido: 1 broker; sin broker no hay ingesta
Target caído (exporter o consumer)                                          up{job=~"kafka-exporter|uso-red"} == 0        1m                                                                                   medido: 1 en ambos; un scrape fallido es binario
          Sensor o agente detenido sum(rate(kafka_topic_partition_current_offset{topic="red-telemetria"}[1m])) < 3        2m                                                      esperado 4 msg/s; < 3 significa que dejó de publicar a

## 7. Estimación de costos y plan de escalado (con lo medido)

In [13]:
pico_eps = max(m["inputRowsPerSecond"] for m in muestras)
print(f"micro-lotes con datos: {sum(1 for _, n, _ in resumen_lotes if n)} de {len(resumen_lotes)}; filas por lote: "
      f"min {min(n for _, n, _ in resumen_lotes if n)}, max {max(n for _, n, _ in resumen_lotes)}")
avg_lat = lat_min["avgLatencyMs"].mean()
gb_mes = mb_h * 730 / 1000
PRECIO_T3_MEDIUM_H, PRECIO_S3_GB_MES, HORAS_MES = 0.0416, 0.023, 730   # AWS us-east-1, referencia de la guía S9

tabla = [
    ("Eventos por segundo (pico)", f"{pico_eps:.2f} eventos/s (inputRowsPerSecond)", "4 sensores x 1 Hz"),
    ("Latencia promedio por minuto", f"{avg_lat:.0f} ms (peor minuto {peor_avg:.0f} ms)", "dominada por el trigger de 5 s"),
    ("Crecimiento del Parquet", f"{filas_h:,.0f} filas/h = {mb_h:.2f} MB/h = {gb_mes:.2f} GB/mes",
     f"S3 Standard: US$ {gb_mes * PRECIO_S3_GB_MES:.3f}/mes por cada mes retenido"),
    ("Capacidad de Spark con backlog", f"{capacidad:,.0f} filas/s", f"holgura = {capacidad / eventos_s:,.0f} veces el ritmo actual"),
    ("Cómputo", "3 instancias t3.medium (Kafka; Spark streaming; Prometheus+Grafana)",
     f"3 x {PRECIO_T3_MEDIUM_H} x {HORAS_MES} = US$ {3 * PRECIO_T3_MEDIUM_H * HORAS_MES:.2f}/mes"),
]
print(pd.DataFrame(tabla, columns=["Variable", "Valor medido", "Supuesto / costo"]).to_string(index=False))

print("\nCosto mensual estimado:")
for meses in (1, 6, 12):
    almacen = gb_mes * meses * PRECIO_S3_GB_MES
    print(f"  retención {meses:>2} meses de Parquet: cómputo US$ {3 * PRECIO_T3_MEDIUM_H * HORAS_MES:.2f} + almacenamiento US$ {almacen:.2f}"
          f" = US$ {3 * PRECIO_T3_MEDIUM_H * HORAS_MES + almacen:.2f}/mes")

print("\nEscenario campus (dimensión U2): cuántas interfaces caben antes de escalar")
for interfaces in (4, 100, 500, 1000):
    eps = interfaces
    print(f"  {interfaces:>5} interfaces -> {eps:>5} eventos/s | {eps / capacidad * 100:5.1f} % de la capacidad medida de Spark"
          f" | Parquet {gb_mes * interfaces / eventos_s:7.1f} GB/mes")

micro-lotes con datos: 88 de 89; filas por lote: min 12, max 20
                      Variable                                                        Valor medido                                 Supuesto / costo
    Eventos por segundo (pico)                                 4.00 eventos/s (inputRowsPerSecond)                                4 sensores x 1 Hz
  Latencia promedio por minuto                                       2486 ms (peor minuto 2607 ms)                   dominada por el trigger de 5 s
       Crecimiento del Parquet                          13,795 filas/h = 16.69 MB/h = 12.19 GB/mes S3 Standard: US$ 0.280/mes por cada mes retenido
Capacidad de Spark con backlog                                                       7,812 filas/s            holgura = 1,953 veces el ritmo actual
                       Cómputo 3 instancias t3.medium (Kafka; Spark streaming; Prometheus+Grafana)                 3 x 0.0416 x 730 = US$ 91.10/mes

Costo mensual estimado:
  retención  1 meses de

### 7.1 Archivos pequeños: cuánto ocupa de verdad la evidencia

Cada trigger de 5 s escribe un archivo por partición. Se compara el Parquet del stream con el mismo contenido compactado en un solo archivo.

In [ ]:
archivos = [f for f in os.listdir(PARQUET) if f.endswith(".parquet")]
bytes_total = sum(os.path.getsize(os.path.join(PARQUET, f)) for f in archivos)
filas = spark.read.parquet(PARQUET).count()
print(f"Parquet del stream: {len(archivos)} archivos, {bytes_total:,} bytes, {bytes_total / len(archivos):,.0f} bytes/archivo, {bytes_total / filas:,.0f} bytes/fila")

COMPACTO = f"{BASE}/red_observabilidad_compacto"
spark.read.parquet(PARQUET).coalesce(1).write.mode("overwrite").parquet(COMPACTO)
bytes_compacto = sum(os.path.getsize(os.path.join(COMPACTO, f)) for f in os.listdir(COMPACTO) if f.endswith(".parquet"))
print(f"Compactado en 1 archivo: {bytes_compacto:,} bytes, {bytes_compacto / filas:,.1f} bytes/fila "
      f"-> {bytes_total / bytes_compacto:,.0f} veces más chico")
print(f"Crecimiento real con compactación: {filas_h * bytes_compacto / filas / 1e6:.2f} MB/hora = "
      f"{filas_h * bytes_compacto / filas * 730 / 1e9:.2f} GB/mes (en vez de {gb_mes:.2f} GB/mes)")

Parquet del stream: 265 archivos, 2,047,879 bytes, 7,728 bytes/archivo, 1,202 bytes/fila
Compactado en 1 archivo: 52,967 bytes, 31.1 bytes/fila -> 39 veces más chico
Crecimiento real con compactación: 0.43 MB/hora = 0.31 GB/mes (en vez de 12.19 GB/mes)
